# Blanket: reproduce the measurement and baseline audit

Nolan McDermott · NFL Big Data Bowl 2027 · October 22, 2026

**Question:** do combine defensive-back movement traits translate to sustained
man-coverage proximity? Current evidence does not establish predictive value.
This notebook reproduces the cohort, nine historical sensor comparisons,
measurement sensitivity, four simple baselines, and aggregate figures.

[Research brief](https://noland.blog/projects/nfl-big-data-bowl/brief/)
· [Methods](https://github.com/NolanMcD/NolanMcD.github.io/blob/main/docs/blanket.md)

Attach the official competition dataset after accepting its rules. Either the
downloaded ZIP or an extracted folder containing all nine CSVs is supported.
On Kaggle, one matching attached folder is detected automatically. Otherwise
set `DATA_PATH` in the first cell, or the `BLANKET_DATA_PATH` environment variable.

Requires Python 3.10+, numpy 2.x, pandas 2.2+, and matplotlib 3.9+; see the
repository requirements. There are no runtime network downloads. No raw data
or stored execution outputs are embedded here. Scripts below are exact,
readable source copies with hashes. The large source cells are an executable
appendix; the findings and tables appear after the run cell.

The extracted-folder route creates a temporary uncompressed ZIP under the
working directory (about the total CSV size). Kaggle must have sufficient
writable disk. All tracking caches and review packets remain in that working
directory. Do not publish them as notebook output datasets.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

DATA_PATH = None  # Set to your ZIP or extracted CSV folder if auto-discovery is ambiguous.
parents = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repository = next((p for p in parents if (p / 'tools/expand-blanket.py').exists()), None)
if os.environ.get('BLANKET_DATA_PATH'):
    DATA_PATH = Path(os.environ['BLANKET_DATA_PATH'])
if DATA_PATH is None and repository and (repository / 'nfl-big-data-bowl-2027.zip').exists():
    DATA_PATH = repository / 'nfl-big-data-bowl-2027.zip'
if DATA_PATH is None:
    matches = sorted({p.parent for p in Path('/kaggle/input').rglob('players.csv')})
    matches = [p for p in matches if (p / 'combine_tracking.csv').exists() and (p / 'game_tracking_2025.csv').exists()]
    if len(matches) != 1:
        raise RuntimeError('Attach one official dataset or set DATA_PATH explicitly.')
    DATA_PATH = matches[0]
DATA_PATH = Path(DATA_PATH).expanduser().resolve()
default_work = (Path('/kaggle/working/blanket-reproduction') if Path('/kaggle').exists()
                else (repository or Path.cwd()) / 'local-data/big-data-bowl/notebook-run')
WORK = Path(os.environ.get('BLANKET_NOTEBOOK_WORKDIR', str(default_work))).resolve()
(WORK / 'tools').mkdir(parents=True, exist_ok=True)
(WORK / '_data').mkdir(exist_ok=True)
# Optional local matplotlib installation; Kaggle normally has this dependency.
if repository and (repository / 'local-data/big-data-bowl/plot-deps').exists():
    extra = str(repository / 'local-data/big-data-bowl/plot-deps')
    sys.path.insert(0, extra)
    os.environ['PYTHONPATH'] = os.pathsep.join([extra, os.environ.get('PYTHONPATH', '')])
print('Working directory:', WORK)


## Exact source appendix

These cells write only the bundled analysis code into the working directory.


In [ ]:
# analyze-blanket.py; SHA-256 of UTF-8 source: 57bc404beb81899131ff944b687bc03426bfedb11f480760e79fded5946e563d
source = (
    '"""Prototype Blanket measurement from locally licensed BDB tracking data."""\n'
    'from pathlib import Path\n'
    'from zipfile import ZipFile\n'
    'import json\n'
    'import pandas as pd\n'
    'import numpy as np\n'
    'import hashlib\n'
    'from html import escape\n'
    'import argparse\n'
    '\n'
    'ROOT = Path(__file__).resolve().parents[1]\n'
    "OUT = ROOT / 'local-data/big-data-bowl'\n"
    'OUT.mkdir(parents=True, exist_ok=True)\n'
    '\n'
    '\n'
    'def main():\n'
    '    parser = argparse.ArgumentParser(description=__doc__)\n'
    "    parser.add_argument('--publish', action='store_true', help='Publish only aggregate results to the website.')\n"
    '    args = parser.parse_args()\n'
    "    archive_path = ROOT / 'nfl-big-data-bowl-2027.zip'\n"
    '    # The cache is private and may only be reused for the identical input archive.\n'
    '    digest = hashlib.sha256()\n'
    "    with archive_path.open('rb') as handle:\n"
    "        for block in iter(lambda: handle.read(1024*1024), b''):\n"
    '            digest.update(block)\n'
    '    fingerprint = digest.hexdigest()\n'
    '    with ZipFile(archive_path) as archive:\n'
    '        def read(name, **kwargs):\n'
    "            return pd.read_csv(archive.open('nfl-big-data-bowl-2027/' + name), **kwargs)\n"
    "        combine = read('combine_results.csv').merge(read('players.csv'), on=['nfl_id', 'draft_year'], validate='one_to_one')\n"
    "        cb = combine[(combine.nfl_position == 'CB') & combine.three_cone.notna()]\n"
    "        pp = read('player_play.csv', low_memory=False)\n"
    "        games = read('games.csv')\n"
    "        regular_ids = games.loc[games.season_type == 'REG','game_id']\n"
    "        eligible = pp[pp.nfl_id.isin(cb.nfl_id) & pp.lined_up_position.isin(['CB', 'SLOT_CB']) &\n"
    "                      (pp.team_coverage_man_zone == 'MAN_COVERAGE') & (pp.coverage_assignment == 'MAN') &\n"
    "                      (pp.play_nullified_by_penalty == 'N') & pp.pass_result.isin(['C', 'I', 'IN']) & pp.game_id.isin(regular_ids)].copy()\n"
    "        eligible.to_csv(OUT / 'eligible-man-reps.csv', index=False)\n"
    "        print('Eligible reps:', len(eligible), 'players:', eligible.nfl_id.nunique(), flush=True)\n"
    '        keys = set(zip(eligible.game_id, eligible.play_id))\n'
    "        cache = OUT / 'candidate-tracking.pkl'\n"
    "        manifest = OUT / 'tracking-cache.json'\n"
    "        cache_key = {'archive_sha256': fingerprint, 'filter_version': 2}\n"
    '        if cache.exists() and manifest.exists() and json.loads(manifest.read_text()) == cache_key:\n'
    '            tracking = pd.read_pickle(cache)\n'
    '        else:\n'
    '            blocks = []\n'
    '            for year in [2023, 2024, 2025]:\n'
    "                print('Reading tracking', year, flush=True)\n"
    "                with archive.open(f'nfl-big-data-bowl-2027/game_tracking_{year}.csv') as raw:\n"
    '                    for chunk in pd.read_csv(raw, chunksize=250000):\n'
    '                        selected = np.fromiter(((g, p) in keys for g, p in zip(chunk.game_id, chunk.play_id)), dtype=bool, count=len(chunk))\n'
    '                        if selected.any():\n'
    '                            blocks.append(chunk.loc[selected])\n'
    '            tracking = pd.concat(blocks, ignore_index=True)\n'
    '            tracking.to_pickle(cache)\n'
    "            manifest.write_text(json.dumps(cache_key), encoding='utf-8')\n"
    "        print('Tracking rows:', len(tracking), 'events:', tracking.event.value_counts().to_dict(), flush=True)\n"
    "        metadata = pp[pp.set_index(['game_id', 'play_id']).index.isin(keys)]\n"
    "        print('Positions:', metadata.lined_up_position.value_counts().to_dict(), flush=True)\n"
    '        diagnostics = []\n'
    "        for (game, play), t in tracking.groupby(['game_id', 'play_id'], sort=False):\n"
    '            meta = metadata[(metadata.game_id == game) & (metadata.play_id == play)]\n'
    '            offense = meta[(meta.team_abbr == meta.possession_team) & meta.route_ran.notna()]\n'
    "            diagnostics.append({'game_id': int(game), 'play_id': int(play), 'tracked_ids': int(t.nfl_id.nunique()),\n"
    "                                'offensive_route_rows': len(offense), 'tracked_receivers': int(offense.nfl_id.isin(t.nfl_id).sum()),\n"
    "                                'snap_events': int((t.event == 'ball_snap').sum()), 'throw_events': int((t.event == 'pass_forward').sum())})\n"
    "        pd.DataFrame(diagnostics).to_csv(OUT / 'tracking-diagnostics.csv', index=False)\n"
    '        print(pd.DataFrame(diagnostics).describe().to_string(), flush=True)\n'
    "        drill = read('combine_tracking.csv')\n"
    '        sensor = sensor_features(drill, cb)\n'
    '        measure(eligible, tracking, metadata, cb, sensor, fingerprint, args.publish)\n'
    '\n'
    '\n'
    'def sensor_features(drill, cb):\n'
    '    """Median speed during fast heading changes, summarized by attempt then player.\n'
    '\n'
    '    Direction changes use circular differences. Speed <1 is excluded to reduce\n'
    '    unstable headings near rest; this is exploratory and not a calibrated trait.\n'
    '    """\n'
    "    selected = drill[(drill.drill_type == 'THREE_CONE_DRILL') &\n"
    "                     (drill.entity_type == 'PLAYER') & drill.nfl_id.isin(cb.nfl_id)]\n"
    '    attempts = []\n'
    "    for (pid,event,attempt), frame in selected.groupby(['nfl_id','event_id','attempt']):\n"
    "        frame = frame.sort_values('time').copy()\n"
    '        seconds = pd.to_datetime(frame.time).diff().dt.total_seconds()\n'
    '        heading = ((frame.dir.diff()+180)%360-180).abs()\n'
    '        turn = (heading / seconds >= 120) & seconds.between(.09,.11) & (frame.s >= 1) & (frame.s.shift(1) >= 1)\n'
    '        if turn.sum() >= 3:\n'
    "            attempts.append({'nfl_id': int(pid), 'event_id': event, 'attempt': int(attempt),\n"
    "                             'turn_speed': float(frame.loc[turn,'s'].median()), 'turn_frames': int(turn.sum())})\n"
    "    table = pd.DataFrame(attempts, columns=['nfl_id','event_id','attempt','turn_speed','turn_frames'])\n"
    "    table.to_csv(OUT/'three-cone-sensor-attempts.csv',index=False)\n"
    "    return table.groupby('nfl_id',as_index=False).agg(turn_speed=('turn_speed','median'),sensor_attempts=('attempt','size'))\n"
    '\n'
    '\n'
    'def measure(eligible, tracking, metadata, cb, sensor, fingerprint, publish):\n'
    '    """Infer pairs at snap only. Outputs are provisional, never charted assignments."""\n'
    "    groups = {k: t for k, t in tracking.groupby(['game_id', 'play_id'], sort=False)}\n"
    "    metas = {k: m for k, m in metadata.groupby(['game_id', 'play_id'], sort=False)}\n"
    '    excluded = {}\n'
    '    reps = []\n'
    '    examples = []\n'
    '    def reject(reason):\n'
    '        excluded[reason] = excluded.get(reason, 0) + 1\n'
    '    for r in eligible.itertuples(index=False):\n'
    '        key = (r.game_id, r.play_id)\n'
    '        t = groups.get(key)\n'
    '        if t is None:\n'
    "            reject('No play tracking'); continue\n"
    "        defender = t[t.nfl_id == r.nfl_id].sort_values('time')\n"
    "        snaps = defender[defender.event == 'ball_snap']\n"
    "        throws = defender[defender.event == 'pass_forward']\n"
    '        if len(snaps) != 1 or len(throws) != 1:\n'
    "            reject('Missing or ambiguous defender snap/throw'); continue\n"
    '        snap, throw = snaps.iloc[0], throws.iloc[0]\n'
    '        start, end = pd.Timestamp(snap.time), pd.Timestamp(throw.time)\n'
    '        duration = (end-start).total_seconds()\n'
    '        if duration < 1:\n'
    "            reject('Less than one second before throw'); continue\n"
    '        meta = metas[key]\n'
    '        offense = meta[(meta.team_abbr == r.possession_team) & meta.route_ran.notna() &\n'
    "                       meta.lined_up_position.isin(['WR', 'SLOT_WR', 'TE', 'RB', 'FB'])]\n"
    '        at_snap = t[(t.time == snap.time) & t.nfl_id.isin(offense.nfl_id)].copy()\n'
    '        if at_snap.empty:\n'
    "            reject('No tracked route runner at snap'); continue\n"
    "        at_snap['distance'] = np.hypot(at_snap.x - snap.x, at_snap.y - snap.y)\n"
    "        at_snap = at_snap.sort_values('distance')\n"
    '        best = at_snap.iloc[0]\n'
    '        if best.distance > 5:\n'
    "            reject('Nearest tracked route runner more than 5 yards at snap'); continue\n"
    '        if len(at_snap) > 1 and at_snap.iloc[1].distance - best.distance < 1:\n'
    "            reject('Ambiguous snap pairing: margin below 1 yard'); continue\n"
    "        receiver = t[t.nfl_id == best.nfl_id].sort_values('time')\n"
    '        # Never switch receivers during a play or select a pair by its outcome.\n'
    "        pair = defender[['time', 'x', 'y']].merge(receiver[['time', 'x', 'y']], on='time',\n"
    "                                               suffixes=('_cb', '_wr'), validate='one_to_one')\n"
    "        pair['timestamp'] = pd.to_datetime(pair.time)\n"
    '        pair = pair[(pair.timestamp >= start) & (pair.timestamp <= end)].copy()\n'
    '        expected = round(duration * 10) + 1\n'
    '        if len(pair) < .9 * expected or pair.empty or pair.timestamp.iloc[0] != start or pair.timestamp.iloc[-1] != end:\n'
    "            reject('Incomplete synchronized window'); continue\n"
    '        if pair.timestamp.diff().dt.total_seconds().max() > .201:\n'
    "            reject('Tracking gap over 0.2 seconds'); continue\n"
    "        if not np.isfinite(pair[['x_cb','y_cb','x_wr','y_wr']].to_numpy()).all():\n"
    "            reject('Nonfinite coordinates'); continue\n"
    "        pair['elapsed'] = (pair.timestamp - start).dt.total_seconds()\n"
    "        pair['distance'] = np.hypot(pair.x_cb - pair.x_wr, pair.y_cb - pair.y_wr)\n"
    '        # Trapezoidal time average handles occasional missing 10 Hz samples.\n'
    '        mean = average_separation(pair.distance.to_numpy(),pair.elapsed.to_numpy())\n'
    '        late = pair[pair.elapsed >= .5]\n'
    '        late_mean = float(np.trapezoid(late.distance, late.elapsed) / (late.elapsed.iloc[-1]-late.elapsed.iloc[0])) if len(late)>1 else None\n'
    "        reps.append({'game_id': int(r.game_id), 'play_id': int(r.play_id), 'nfl_id': int(r.nfl_id),\n"
    "                     'receiver_id': int(best.nfl_id), 'mean_separation': mean, 'late_mean_separation': late_mean,\n"
    "                     'throw_separation': float(pair.distance.iloc[-1]), 'duration': duration,\n"
    "                     'snap_cushion': float(best.distance), 'blanket': is_blanket(mean),\n"
    "                     'observed_runner_count': len(at_snap), 'frames': len(pair)})\n"
    '        if len(examples) < 5:\n'
    '            examples.append((reps[-1], pair.copy(), r.play_direction))\n'
    '    data = pd.DataFrame(reps)\n'
    '    # A receiver cannot be assigned to two measured cornerbacks on the same rep.\n'
    '    if not data.empty:\n'
    "        collision = data.duplicated(['game_id','play_id','receiver_id'],keep=False)\n"
    '        if collision.any():\n'
    "            excluded['Receiver paired to multiple cornerbacks'] = int(collision.sum())\n"
    '            data = data[~collision].copy()\n'
    "    data.to_csv(OUT / 'inferred-coverage-reps.csv', index=False)\n"
    '    if data.empty:\n'
    "        raise RuntimeError('No usable inferred pairs. Do not publish Blanket estimates.')\n"
    "    player = data.groupby('nfl_id').agg(reps=('blanket','size'), blankets=('blanket','sum'),\n"
    "                                       mean_separation=('mean_separation','mean'),\n"
    "                                       snap_cushion=('snap_cushion','mean')).reset_index()\n"
    "    player['blanket_rate'] = player.blankets / player.reps\n"
    "    player = player.merge(cb[['nfl_id','display_name','three_cone','forty','draft_year']], on='nfl_id', validate='one_to_one')\n"
    "    player = player.merge(sensor,on='nfl_id',how='left',validate='one_to_one')\n"
    "    player.to_csv(OUT / 'provisional-player-summary.csv', index=False)\n"
    '    rng = np.random.default_rng(2027)\n'
    '    def correlation(frame):\n'
    '        if len(frame)<4 or frame.three_cone.nunique()<2 or frame.mean_separation.nunique()<2:\n'
    '            return None\n'
    '        return float(np.corrcoef(frame.three_cone, frame.mean_separation)[0,1])\n'
    '    stable = player[player.reps >= 10]\n'
    "    sensor_stable = stable.dropna(subset=['turn_speed'])\n"
    '    bootstrap = []\n'
    '    for _ in range(2000):\n'
    '        v = correlation(stable.iloc[rng.integers(0,len(stable),len(stable))]) if len(stable) else None\n'
    '        if v is not None and np.isfinite(v): bootstrap.append(v)\n'
    '    counts = []\n'
    '    for threshold in [.5,.75,1,1.25,1.5,2,3]:\n'
    "        counts.append({'threshold': threshold, 'count': int((data.mean_separation < threshold).sum()),\n"
    "                       'late_count': int((data.late_mean_separation < threshold).sum())})\n"
    '    bins = [0,.5,1,1.5,2,3,5,8,12,1000]\n'
    '    hist, edges = np.histogram(data.mean_separation, bins=bins)\n'
    "    summary = {'status': 'Exploratory inferred pairings; assignments not independently verified',\n"
    "               'definition': 'Blanket: time-average separation strictly below 1 yard from snap through pass_forward.',\n"
    "               'candidate_reps': len(eligible), 'candidate_players': int(eligible.nfl_id.nunique()),\n"
    "               'screened_reps': len(data), 'screened_players': int(data.nfl_id.nunique()),\n"
    "               'provisional_blankets': int(data.blanket.sum()), 'blanket_rate': float(data.blanket.mean()),\n"
    "               'mean_separation': float(data.mean_separation.mean()),\n"
    "               'median_separation': float(data.mean_separation.median()), 'exclusions': excluded,\n"
    "               'single_observed_runner_reps': int((data.observed_runner_count == 1).sum()),\n"
    "               'correlation': correlation(stable), 'correlation_players': len(stable),\n"
    "               'correlation_min_reps': 10,\n"
    "               'correlation_bootstrap_95': [float(v) for v in np.quantile(bootstrap,[.025,.975])] if bootstrap else None,\n"
    "               'thresholds': counts,\n"
    "               'histogram': [{'label': f'{bins[i]:g}–{bins[i+1]:g}' if i<len(hist)-1 else '12+', 'count': int(n)} for i,n in enumerate(hist)],\n"
    "               'window_sensitivity': {'after_half_second_blankets': int((data.late_mean_separation < 1).sum()),\n"
    "                                      'throw_only_below_one': int((data.throw_separation < 1).sum())},\n"
    "               'sensor': {'players_with_feature': len(sensor), 'comparison_players': len(sensor_stable),\n"
    "                          'correlation_with_mean_separation': float(np.corrcoef(sensor_stable.turn_speed,sensor_stable.mean_separation)[0,1]) if len(sensor_stable)>=4 else None,\n"
    "                          'definition': 'Median speed during heading changes of at least 120 degrees/second; adjacent speeds >=1; median across eligible attempts.'},\n"
    "               'data_source': 'NFL / Next Gen Stats, NFL Big Data Bowl 2027 on Kaggle',\n"
    "               'analysis_version': 3,\n"
    "               'season_scope': 'Regular season only, 2023–2025',\n"
    "               'input_sha256': fingerprint}\n"
    '    assert len(data)+sum(excluded.values()) == len(eligible)\n'
    '    assert int(hist.sum()) == len(data)\n'
    "    assert not data.duplicated(['game_id','play_id','nfl_id']).any()\n"
    '    assert data.blanket.equals(data.mean_separation < 1)\n'
    "    (OUT/'blanket-summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')\n"
    '    write_review(examples, player)\n'
    '    if publish:\n'
    '        # No frame trajectories, player identities, stopwatch inputs, or rep rows.\n'
    "        (ROOT/'_data/blanket_results.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')\n"
    '    print(json.dumps(summary,indent=2),flush=True)\n'
    '\n'
    '\n'
    'def average_separation(distance, seconds):\n'
    '    distance, seconds = np.asarray(distance,dtype=float), np.asarray(seconds,dtype=float)\n'
    '    if len(distance) != len(seconds) or len(seconds)<2 or not np.isfinite(distance).all() or not np.isfinite(seconds).all():\n'
    "        raise ValueError('A complete finite distance/time series is required')\n"
    '    if (distance<0).any() or (np.diff(seconds)<=0).any():\n'
    "        raise ValueError('Distances must be nonnegative and timestamps strictly increasing')\n"
    '    return float(np.trapezoid(distance,seconds)/(seconds[-1]-seconds[0]))\n'
    '\n'
    '\n'
    'def is_blanket(mean_separation):\n'
    '    if not np.isfinite(mean_separation) or mean_separation<0:\n'
    "        raise ValueError('Blanket requires a finite nonnegative measurement')\n"
    '    return bool(mean_separation < 1)\n'
    '\n'
    '\n'
    'def write_review(examples, player):\n'
    '    cards = []\n'
    '    for rep, p, direction in examples:\n'
    '        xmin, xmax = min(p.x_cb.min(),p.x_wr.min())-2, max(p.x_cb.max(),p.x_wr.max())+2\n'
    '        ymin, ymax = min(p.y_cb.min(),p.y_wr.min())-2, max(p.y_cb.max(),p.y_wr.max())+2\n'
    '        scale = min(500/(xmax-xmin),250/(ymax-ymin))\n'
    "        point = lambda x,y: f'{30+(x-xmin)*scale:.1f},{280-(y-ymin)*scale:.1f}'\n"
    '        paths = \'\'.join(f\'<polyline points="{" ".join(point(x,y) for x,y in zip(p[f"x_{side}"],p[f"y_{side}"]))}" fill="none" stroke="{color}" stroke-width="3"/>\' for side,color in [(\'cb\',\'#6ee7b7\'),(\'wr\',\'#fbbf24\')])\n'
    '        markers = \'\'.join(f\'<circle cx="{point(p[f"x_{side}"].iloc[0],p[f"y_{side}"].iloc[0]).split(",")[0]}" cy="{point(p[f"x_{side}"].iloc[0],p[f"y_{side}"].iloc[0]).split(",")[1]}" r="5" fill="{color}"/>\' for side,color in [(\'cb\',\'#6ee7b7\'),(\'wr\',\'#fbbf24\')])\n'
    '        curve = \' \'.join(f\'{30+float(t)/rep["duration"]*500:.1f},{260-float(d)/max(3,p.distance.max())*210:.1f}\' for t,d in zip(p.elapsed,p.distance))\n'
    '        threshold_y = 260-210/max(3,p.distance.max())\n'
    '        cards.append(f\'<article><h2>Game {rep["game_id"]} · Play {rep["play_id"]}</h2><p>CB {rep["nfl_id"]} / inferred receiver {rep["receiver_id"]}. Mean {rep["mean_separation"]:.2f} yards. Blanket: {rep["blanket"]}. Offense moves {escape(direction)}. Dots mark snap; trajectories end at throw. Equal yard scale on both field axes.</p><svg viewBox="0 0 560 310" role="img" aria-label="Inferred cornerback and receiver trajectories">{paths}{markers}</svg><svg viewBox="0 0 560 310" role="img" aria-label="Separation from snap to throw"><line x1="30" x2="530" y1="{threshold_y}" y2="{threshold_y}" stroke="#fbbf24" stroke-dasharray="5 5"/><polyline points="{curve}" fill="none" stroke="#6ee7b7" stroke-width="3"/><text x="30" y="300" fill="white">Snap → throw · dashed line = 1 yard · max {p.distance.max():.1f} yards</text></svg></article>\')\n'
    '    html = \'<!doctype html><html lang="en"><meta charset="utf-8"><title>Private Blanket matchup review</title><style>body{background:#101c23;color:#eee;font:16px system-ui;max-width:1150px;margin:40px auto;padding:20px}article{border-top:1px solid #567;padding:20px 0}svg{width:48%;background:#16372d;border-radius:12px}p{line-height:1.7}</style><h1>Five inferred matchup examples</h1><p>Private review of supplied tracking. Green = CB. Gold = inferred receiver. These pairings are algorithmic and are not confirmed coverage assignments. Missing players can invalidate a pairing. Do not redistribute this review as raw tracking.</p>\' + \'\'.join(cards) + \'</html>\'\n'
    "    (OUT/'matchup-review.html').write_text(html,encoding='utf-8')\n"
    '\n'
    '\n'
    "if __name__ == '__main__':\n"
    '    main()\n'
)
assert hashlib.sha256(source.encode('utf-8')).hexdigest() == '57bc404beb81899131ff944b687bc03426bfedb11f480760e79fded5946e563d'
(WORK / 'tools' / 'analyze-blanket.py').write_bytes(source.encode('utf-8'))


In [ ]:
# expand-blanket.py; SHA-256 of UTF-8 source: 7bb039dfe0f58589d347c8423b946f904a1ce29d58dbab1918b52ef74fd43147
source = (
    '"""Expanded, provisional combine-to-coverage study. Raw inputs stay private."""\n'
    'import argparse\n'
    'import hashlib\n'
    'import importlib.util\n'
    'import json\n'
    'from collections import Counter\n'
    'from pathlib import Path\n'
    'from zipfile import ZipFile\n'
    '\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    '\n'
    'ROOT = Path(__file__).resolve().parents[1]\n'
    "OUT = ROOT/'local-data/big-data-bowl/expanded'\n"
    'OUT.mkdir(parents=True,exist_ok=True)\n'
    "spec = importlib.util.spec_from_file_location('blanket',ROOT/'tools/analyze-blanket.py')\n"
    'blanket = importlib.util.module_from_spec(spec)\n'
    'spec.loader.exec_module(blanket)\n'
    "DRILLS = {'W_DRILL':'W drill', 'BACK_PEDAL_AND_90_DEGREE_BREAK_TO_BALL':'90° backpedal / break',\n"
    "          'BACK_PEDAL_AND_TRANSITION_45_DEGREE_REACTION':'45° backpedal / transition'}\n"
    '\n'
    '\n'
    'def phase_features(frame):\n'
    '    """Measure the largest moving heading change, requiring observed flanking data.\n'
    '\n'
    '    This is an operational dominant-turn window, not a coach-annotated drill phase.\n'
    '    Use a fixed 0.4-second circular heading difference and +/-0.6 second flanks.\n'
    '    """\n'
    "    f=frame.sort_values('time').drop_duplicates('time').copy()\n"
    '    t=(pd.to_datetime(f.time)-pd.to_datetime(f.time.iloc[0])).dt.total_seconds().to_numpy()\n'
    '    speed=f.s.rolling(3,center=True,min_periods=1).median().to_numpy()\n'
    '    radians=np.deg2rad(f.dir.to_numpy())\n'
    '    sin=pd.Series(np.sin(radians)).rolling(3,center=True,min_periods=1).mean()\n'
    '    cos=pd.Series(np.cos(radians)).rolling(3,center=True,min_periods=1).mean()\n'
    '    heading=np.arctan2(sin,cos).to_numpy()\n'
    '    if len(f)<15 or not np.isfinite(np.column_stack([t,speed,heading])).all() or np.diff(t).max()>.201:\n'
    '        return None\n'
    '    swings=np.full(len(f),np.nan)\n'
    '    for i in range(2,len(f)-2):\n'
    '        if t[i]-t[0]>=.6 and t[-1]-t[i]>=.6 and speed[i]>=1:\n'
    '            swings[i]=abs(np.arctan2(np.sin(heading[i+2]-heading[i-2]),np.cos(heading[i+2]-heading[i-2])))\n'
    '    if not np.isfinite(swings).any() or np.nanmax(swings)<1e-6:return None\n'
    '    i=int(np.nanargmax(swings))\n'
    '    center=t[i]\n'
    '    turn=np.flatnonzero((t>=center-.2-1e-6)&(t<=center+.2+1e-6))\n'
    '    low=int(turn[np.argmin(speed[turn])])\n'
    '    before=np.flatnonzero((t>=center-.6-1e-6)&(t<=t[low]-.1+1e-6))\n'
    '    after=np.flatnonzero((t>=t[low]+.1-1e-6)&(t<=center+.6+1e-6))\n'
    '    if len(before)==0 or len(after)==0:return None\n'
    '    pre=int(before[np.argmax(speed[before])]); post=int(after[np.argmax(speed[after])])\n'
    "    return {'turn_speed':float(speed[low]), 'braking_loss':float((speed[pre]-speed[low])/(t[low]-t[pre])),\n"
    "            'reacceleration':float((speed[post]-speed[low])/(t[post]-t[low])),\n"
    "            'heading_change_degrees':float(np.rad2deg(swings[i])), 'center_seconds':float(center),\n"
    "            'duration':float(t[-1]),'turn_seconds':float(t[low]),'pre_seconds':float(t[pre]),'post_seconds':float(t[post])}\n"
    '\n'
    '\n'
    'def drill_table(drill,cb_ids):\n'
    "    selected=drill[drill.nfl_id.isin(cb_ids)&(drill.entity_type=='PLAYER')&drill.drill_name.isin(DRILLS)]\n"
    '    attempts=[]; rejected=Counter()\n'
    "    for (pid,name,event,attempt),f in selected.groupby(['nfl_id','drill_name','event_id','attempt']):\n"
    '        features=phase_features(f)\n'
    '        if features is None:rejected[name]+=1;continue\n'
    "        attempts.append({'nfl_id':int(pid),'drill':name,'event_id':event,'attempt':int(attempt),**features})\n"
    '    a=pd.DataFrame(attempts)\n'
    "    a.to_csv(OUT/'drill-attempt-features.csv',index=False)\n"
    "    features=['turn_speed','braking_loss','reacceleration','heading_change_degrees']\n"
    "    players=a.groupby(['nfl_id','drill'],as_index=False)[features].median()\n"
    '    return a,players,dict(rejected)\n'
    '\n'
    '\n'
    'def ridge_predict(train_x,train_y,test_x,penalty=1.):\n'
    '    # All imputation and scaling are learned inside the held-out-player fold.\n'
    '    med=np.nanmedian(train_x,axis=0)\n'
    '    med=np.where(np.isfinite(med),med,0.)\n'
    '    x=np.where(np.isfinite(train_x),train_x,med)\n'
    '    z=np.where(np.isfinite(test_x),test_x,med)\n'
    '    mean=x.mean(axis=0);std=x.std(axis=0);std=np.where(std>1e-8,std,1.)\n'
    '    design=np.column_stack([np.ones(len(x)),(x-mean)/std])\n'
    '    target=np.column_stack([np.ones(len(z)),(z-mean)/std])\n'
    '    regularizer=np.diag([0.]+[penalty]*x.shape[1])\n'
    '    beta=np.linalg.solve(design.T@design+regularizer,design.T@train_y)\n'
    '    return target@beta\n'
    '\n'
    '\n'
    'def compare_model(players,feature):\n'
    '    """One fixed extra drill feature, evaluated on entirely unseen CBs."""\n'
    '    p=players[players.reps>=10].dropna(subset=[feature]).copy()\n'
    "    if len(p)<12:return {'players':len(p),'status':'Too few players for this comparison'}\n"
    "    baseline=['snap_cushion','forty','combine_height','combine_weight']\n"
    '    y=p.mean_separation.to_numpy()\n'
    "    predictions={name:[] for name in ['context','context_plus_drill']}\n"
    '    errors=[]\n'
    '    for i in range(len(p)):\n'
    '        keep=np.arange(len(p))!=i\n'
    "        for name,columns in [('context',baseline),('context_plus_drill',baseline+[feature])]:\n"
    '            estimate=ridge_predict(p[columns].to_numpy()[keep],y[keep],p[columns].to_numpy()[[i]])[0]\n'
    '            predictions[name].append(float(estimate))\n'
    "        errors.append(abs(y[i]-predictions['context'][i])-abs(y[i]-predictions['context_plus_drill'][i]))\n"
    '    improvement=np.asarray(errors)\n'
    '    rng=np.random.default_rng(20271018)\n'
    '    boot=np.array([rng.choice(improvement,size=len(improvement),replace=True).mean() for _ in range(2000)])\n'
    "    result={'players':len(p),'status':'Exploratory leave-one-player-out comparison',\n"
    "            'baseline_mae':float(np.mean(np.abs(y-predictions['context']))),\n"
    "            'augmented_mae':float(np.mean(np.abs(y-predictions['context_plus_drill']))),\n"
    "            'mae_improvement':float(improvement.mean()),'improvement_95':np.quantile(boot,[.025,.975]).tolist(),\n"
    "            'players_missing_forty':int(p.forty.isna().sum())}\n"
    '    return result\n'
    '\n'
    '\n'
    'def main():\n'
    '    parser=argparse.ArgumentParser(description=__doc__)\n'
    "    parser.add_argument('--publish',action='store_true')\n"
    '    args=parser.parse_args()\n'
    '    digest=hashlib.sha256()\n'
    "    with (ROOT/'nfl-big-data-bowl-2027.zip').open('rb') as handle:\n"
    "        for b in iter(lambda:handle.read(1024*1024),b''):digest.update(b)\n"
    '    fingerprint=digest.hexdigest()\n'
    "    with ZipFile(ROOT/'nfl-big-data-bowl-2027.zip') as archive:\n"
    "        def read(name,**kwargs):return pd.read_csv(archive.open('nfl-big-data-bowl-2027/'+name),**kwargs)\n"
    "        combine=read('combine_results.csv').merge(read('players.csv'),on=['nfl_id','draft_year'],validate='one_to_one')\n"
    "        cb=combine[combine.nfl_position=='CB']\n"
    "        pp=read('player_play.csv',low_memory=False)\n"
    "        games=read('games.csv')\n"
    "        regular=games.loc[games.season_type=='REG','game_id']\n"
    "        eligible=pp[pp.nfl_id.isin(cb.nfl_id)&pp.game_id.isin(regular)&pp.lined_up_position.isin(['CB','SLOT_CB'])&\n"
    "                    (pp.team_coverage_man_zone=='MAN_COVERAGE')&(pp.coverage_assignment=='MAN')&\n"
    "                    (pp.play_nullified_by_penalty=='N')&pp.pass_result.isin(['C','I','IN'])].copy()\n"
    "        eligible.to_csv(OUT/'candidate-reps.csv',index=False)\n"
    "        print('Expanded candidates:',len(eligible),'players',eligible.nfl_id.nunique(),flush=True)\n"
    '        keys=set(zip(eligible.game_id,eligible.play_id))\n'
    "        cache=OUT/'tracking.pkl';manifest=OUT/'tracking-cache.json'\n"
    "        signature={'input_sha256':fingerprint,'filter_version':1}\n"
    '        if cache.exists() and manifest.exists() and json.loads(manifest.read_text())==signature:\n'
    '            tracking=pd.read_pickle(cache)\n'
    '        else:\n'
    '            parts=[]\n'
    '            for year in [2023,2024,2025]:\n'
    "                print('Reading expanded tracking',year,flush=True)\n"
    "                for chunk in read(f'game_tracking_{year}.csv',chunksize=350000):\n"
    '                    mask=np.fromiter(((g,p) in keys for g,p in zip(chunk.game_id,chunk.play_id)),dtype=bool,count=len(chunk))\n'
    '                    if mask.any():parts.append(chunk.loc[mask])\n'
    '            tracking=pd.concat(parts,ignore_index=True)\n'
    "            tracking.to_pickle(cache);manifest.write_text(json.dumps(signature),encoding='utf-8')\n"
    "        relevant=pp[pp.set_index(['game_id','play_id']).index.isin(keys)]\n"
    "        tg={k:g for k,g in tracking.groupby(['game_id','play_id'],sort=False)}\n"
    "        mg={k:g for k,g in relevant.groupby(['game_id','play_id'],sort=False)}\n"
    '        measures=[];exclusions=Counter()\n'
    '        for r in eligible.itertuples(index=False):\n'
    '            t=tg.get((r.game_id,r.play_id));m=mg[(r.game_id,r.play_id)]\n'
    '            rep,reason=measure_pair(r,t,m)\n'
    '            if rep is None:exclusions[reason]+=1\n'
    '            else:measures.append(rep)\n'
    '        reps=pd.DataFrame(measures)\n'
    "        collision=reps.duplicated(['game_id','play_id','receiver_id'],keep=False)\n"
    "        if collision.any():exclusions['Receiver selected by multiple measured CBs']+=int(collision.sum());reps=reps[~collision].copy()\n"
    "        reps['blanket']=reps.mean_separation.map(blanket.is_blanket)\n"
    "        reps.to_csv(OUT/'coverage-reps.csv',index=False)\n"
    "        players=reps.groupby('nfl_id',as_index=False).agg(reps=('blanket','size'),blankets=('blanket','sum'),\n"
    "                      mean_separation=('mean_separation','mean'),snap_cushion=('snap_cushion','mean'))\n"
    "        players=players.merge(cb[['nfl_id','display_name','draft_year','three_cone','forty','combine_height','combine_weight']],on='nfl_id',validate='one_to_one')\n"
    "        attempts,features,drill_rejects=drill_table(read('combine_tracking.csv'),cb.nfl_id)\n"
    '        models=[]\n'
    '        for name,label in DRILLS.items():\n'
    '            f=features[features.drill==name]\n'
    "            joined=players.merge(f,on='nfl_id',validate='one_to_one')\n"
    "            for feature in ['turn_speed','braking_loss','reacceleration']:\n"
    '                result=compare_model(joined,feature)\n'
    "                models.append({'drill':label,'feature':feature,**result})\n"
    "        players.to_csv(OUT/'players.csv',index=False)\n"
    "        feature_availability=[{'drill':label,'players_with_tracking':int(features.loc[features.drill==name,'nfl_id'].nunique()),\n"
    "                              'players_with_coverage':int(players.nfl_id.isin(features.loc[features.drill==name,'nfl_id']).sum())} for name,label in DRILLS.items()]\n"
    '        summary=summarize(reps,players,eligible,exclusions)\n'
    '        review=write_review_sample(reps,tg,mg)\n'
    "        summary.update({'analysis_version':4,'updated':'October 18, 2026','input_sha256':fingerprint,\n"
    "                        'features':feature_availability,'models':models,'drill_attempt_exclusions':drill_rejects,\n"
    "                        'review':review})\n"
    "        (OUT/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')\n"
    "        if args.publish:(ROOT/'_data/blanket_expanded.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')\n"
    '        print(json.dumps(summary,indent=2),flush=True)\n'
    '\n'
    '\n'
    'def measure_pair(r,t,metadata):\n'
    "    if t is None:return None,'No play tracking'\n"
    "    defender=t[t.nfl_id==r.nfl_id].sort_values('time')\n"
    "    snap=defender[defender.event=='ball_snap'];throw=defender[defender.event=='pass_forward']\n"
    "    if len(snap)!=1 or len(throw)!=1:return None,'Missing or ambiguous snap/throw'\n"
    '    snap,throw=snap.iloc[0],throw.iloc[0]\n'
    '    start,end=pd.Timestamp(snap.time),pd.Timestamp(throw.time)\n'
    '    duration=(end-start).total_seconds()\n'
    "    if duration<1:return None,'Less than one second before throw'\n"
    "    off=metadata[(metadata.team_abbr==r.possession_team)&metadata.route_ran.notna()&metadata.lined_up_position.isin(['WR','SLOT_WR','TE','RB','FB'])]\n"
    '    runners=t[(t.time==snap.time)&t.nfl_id.isin(off.nfl_id)].copy()\n'
    "    if runners.empty:return None,'No tracked route runner at snap'\n"
    "    runners['distance']=np.hypot(runners.x-snap.x,runners.y-snap.y)\n"
    "    runners=runners.sort_values('distance');best=runners.iloc[0]\n"
    "    if best.distance>5:return None,'Nearest tracked route runner more than 5 yards at snap'\n"
    "    if len(runners)>1 and runners.iloc[1].distance-best.distance<1:return None,'Ambiguous receiver at snap'\n"
    "    other_ids=metadata.loc[(metadata.team_abbr==r.team_abbr)&(metadata.coverage_assignment=='MAN')&(metadata.nfl_id!=r.nfl_id),'nfl_id']\n"
    '    others=t[(t.time==snap.time)&t.nfl_id.isin(other_ids)]\n'
    '    competing=np.hypot(others.x-best.x,others.y-best.y)\n'
    '    # This narrower sensitivity still cannot account for unobserved defenders.\n'
    '    mutual=bool(len(competing)>0 and competing.min()-best.distance>=1)\n'
    "    runner=t[t.nfl_id==best.nfl_id].sort_values('time')\n"
    "    if defender.time.duplicated().any() or runner.time.duplicated().any():return None,'Duplicate player timestamps'\n"
    "    pair=defender[['time','x','y']].merge(runner[['time','x','y']],on='time',suffixes=('_cb','_wr'),validate='one_to_one')\n"
    "    pair['timestamp']=pd.to_datetime(pair.time,format='ISO8601')\n"
    '    pair=pair[(pair.timestamp>=start)&(pair.timestamp<=end)].copy()\n'
    '    expected=round(duration*10)+1\n'
    "    if pair.empty or len(pair)<.9*expected or pair.timestamp.iloc[0]!=start or pair.timestamp.iloc[-1]!=end:return None,'Incomplete synchronized window'\n"
    "    if pair.timestamp.diff().dt.total_seconds().max()>.201:return None,'Gap over 0.2 seconds'\n"
    "    if not np.isfinite(pair[['x_cb','y_cb','x_wr','y_wr']].to_numpy()).all():return None,'Nonfinite coordinates'\n"
    "    pair['elapsed']=(pair.timestamp-start).dt.total_seconds()\n"
    "    pair['distance']=np.hypot(pair.x_cb-pair.x_wr,pair.y_cb-pair.y_wr)\n"
    '    late=pair[pair.elapsed>=.5]\n'
    '    receiver_meta=off[off.nfl_id==best.nfl_id].iloc[0]\n'
    "    return {'game_id':int(r.game_id),'play_id':int(r.play_id),'nfl_id':int(r.nfl_id),'receiver_id':int(best.nfl_id),\n"
    "            'mean_separation':blanket.average_separation(pair.distance.to_numpy(),pair.elapsed.to_numpy()),\n"
    "            'late_mean_separation':blanket.average_separation(late.distance.to_numpy(),late.elapsed.to_numpy()),\n"
    "            'throw_separation':float(pair.distance.iloc[-1]),'snap_cushion':float(best.distance),'duration':duration,\n"
    "            'route':receiver_meta.route_ran,'observed_runners':len(runners),'mutual_observed_margin':mutual,\n"
    "            'observed_other_man_defenders':len(others),'frames':len(pair)},None\n"
    '\n'
    '\n'
    'def summarize(reps,players,eligible,exclusions):\n'
    '    assert len(reps)+sum(exclusions.values())==len(eligible)\n'
    "    assert not reps.duplicated(['game_id','play_id','nfl_id']).any()\n"
    "    bins=[0,.5,1,1.5,2,3,5,8,12,float('inf')]\n"
    '    hist,_=np.histogram(reps.mean_separation,bins=bins)\n'
    "    threshold=[{'threshold':v,'count':int((reps.mean_separation<v).sum()),'late_count':int((reps.late_mean_separation<v).sum())} for v in [.5,.75,1,1.25,1.5,2,3]]\n"
    '    strata=[]\n'
    "    for label,selected in [('Starting cushion <2 yd',reps[reps.snap_cushion<2]),('Starting cushion 2–3 yd',reps[reps.snap_cushion.between(2,3,inclusive='left')]),('Starting cushion 3–5 yd',reps[reps.snap_cushion>=3]),('Observed reciprocal margin',reps[reps.mutual_observed_margin])]:\n"
    "        strata.append({'label':label,'reps':len(selected),'players':int(selected.nfl_id.nunique()),'blankets':int(selected.blanket.sum()),\n"
    "                       'mean_separation':float(selected.mean_separation.mean()) if len(selected) else None})\n"
    "    return {'status':'Expanded exploratory sample; assignments inferred, not film verified',\n"
    "            'candidate_reps':len(eligible),'candidate_players':int(eligible.nfl_id.nunique()),'screened_reps':len(reps),\n"
    "            'screened_players':len(players),'provisional_blankets':int(reps.blanket.sum()),'blanket_rate':float(reps.blanket.mean()),\n"
    "            'exclusions':dict(exclusions),'single_observed_runner_reps':int((reps.observed_runners==1).sum()),\n"
    "            'median_separation':float(reps.mean_separation.median()),'thresholds':threshold,\n"
    "            'histogram':[{'label':f'{bins[i]:g}–{bins[i+1]:g}' if i<8 else '12+','count':int(n)} for i,n in enumerate(hist)],\n"
    "            'window_sensitivity':{'after_half_second_blankets':int((reps.late_mean_separation<1).sum()),'throw_only_below_one':int((reps.throw_separation<1).sum())},\n"
    "            'strata':strata,'players_min_ten_reps':int((players.reps>=10).sum()),'data_source':'NFL / Next Gen Stats, Big Data Bowl 2027 on Kaggle',\n"
    "            'season_scope':'Regular season 2023–2025','definition':'Average separation strictly below 1 yard, snap through throw'}\n"
    '\n'
    '\n'
    'def write_review_sample(reps,tg,mg):\n'
    '    # Deliberately include every positive plus near-threshold, high-distance,\n'
    '    # reciprocal-margin, and random ordinary reps. This is not a random accuracy sample.\n'
    '    selected=reps[reps.blanket].copy()\n'
    "    extra=[reps.iloc[(reps.mean_separation-1).abs().argsort()[:8]],reps.nlargest(5,'mean_separation'),\n"
    '           reps[reps.mutual_observed_margin].sample(n=min(5,int(reps.mutual_observed_margin.sum())),random_state=2027),\n'
    '           reps.sample(n=min(30,len(reps)),random_state=2027)]\n'
    "    sample=pd.concat([selected,*extra]).drop_duplicates(['game_id','play_id','nfl_id']).head(30).copy()\n"
    "    sample['review_id']=[f'R{i+1:02}' for i in range(len(sample))]\n"
    "    sample['tracking_review']='pending';sample['film_review']='not verified';sample['review_notes']=''\n"
    "    notes_path=OUT/'tracking-review-notes.json'\n"
    "    notes=json.loads(notes_path.read_text(encoding='utf-8')) if notes_path.exists() else {}\n"
    '    for index,r in sample.iterrows():\n'
    "        key=f'{int(r.game_id)}/{int(r.play_id)}/{int(r.nfl_id)}/{int(r.receiver_id)}'\n"
    '        note=notes.get(key)\n'
    '        if note:\n'
    "            sample.at[index,'tracking_review']=note['status']\n"
    "            sample.at[index,'review_notes']=note['note']\n"
    "    sample.to_csv(OUT/'matchup-review.csv',index=False)\n"
    '    panels=[]\n'
    '    for r in sample.itertuples(index=False):\n'
    '        t=tg[(r.game_id,r.play_id)]\n'
    "        d=t[t.nfl_id==r.nfl_id].sort_values('time');s=d[d.event=='ball_snap'].iloc[0];e=d[d.event=='pass_forward'].iloc[0]\n"
    "        w=t[t.nfl_id==r.receiver_id].sort_values('time')\n"
    "        pair=d[['time','x','y']].merge(w[['time','x','y']],on='time',suffixes=('_cb','_wr'))\n"
    '        pair=pair[pair.time.between(s.time,e.time)].copy()\n'
    "        pair['seconds']=(pd.to_datetime(pair.time)-pd.Timestamp(s.time)).dt.total_seconds()\n"
    "        pair['distance']=np.hypot(pair.x_cb-pair.x_wr,pair.y_cb-pair.y_wr)\n"
    "        panels.append({'review_id':r.review_id,'game_id':r.game_id,'play_id':r.play_id,'cb':r.nfl_id,'receiver':r.receiver_id,\n"
    "                       'mean':r.mean_separation,'blanket':r.blanket,'mutual':r.mutual_observed_margin,\n"
    "                       'route':r.route,'observed_runners':r.observed_runners,'frames':pair[['seconds','x_cb','y_cb','x_wr','y_wr','distance']].to_dict('records')})\n"
    "    (OUT/'review-panels.json').write_text(json.dumps(panels,indent=2),encoding='utf-8')\n"
    "    reviewed=int((sample.tracking_review!='pending').sum())\n"
    "    return {'prepared':len(sample),'tracking_reviewed':reviewed,'flagged_for_film':int((sample.tracking_review=='flagged for film').sum()),\n"
    "            'positive_reps_in_review':int(sample.blanket.sum()),'film_verified':0,\n"
    "            'status':'Tracking-only visual consistency review; no assignment truth labels',\n"
    "            'sampling':'Outcome-enriched: positives, near-threshold reps, large gaps, reciprocal pairs, and ordinary reps. Not an accuracy sample.',\n"
    "            'flag_policy':'Flagged reps remain in the primary analysis; excluding poor outcomes after review would bias the result.'}\n"
    '\n'
    '\n'
    "if __name__=='__main__':main()\n"
)
assert hashlib.sha256(source.encode('utf-8')).hexdigest() == '7bb039dfe0f58589d347c8423b946f904a1ce29d58dbab1918b52ef74fd43147'
(WORK / 'tools' / 'expand-blanket.py').write_bytes(source.encode('utf-8'))


In [ ]:
# validate-blanket.py; SHA-256 of UTF-8 source: 5004a99cede73dd16d147825a32cfbb185b31d4daf89748eae06ab087d97bf77
source = (
    '"""Audit Blanket measurement without fitting or selecting prediction models.\n'
    '\n'
    'Requires expand-blanket.py outputs from the identical archive. Review packets and\n'
    'annotations stay private; --publish writes aggregate evidence only.\n'
    '"""\n'
    'import argparse\n'
    'import hashlib\n'
    'import importlib.util\n'
    'import json\n'
    'from collections import Counter\n'
    'from html import escape\n'
    'from pathlib import Path\n'
    'from zipfile import ZipFile\n'
    '\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    '\n'
    'ROOT = Path(__file__).resolve().parents[1]\n'
    "INPUT = ROOT / 'local-data/big-data-bowl/expanded'\n"
    "OUT = INPUT / 'validation'\n"
    "KEYS = ['game_id', 'play_id', 'nfl_id', 'receiver_id']\n"
    'SEED = 20261022\n'
    "spec = importlib.util.spec_from_file_location('expanded', ROOT / 'tools/expand-blanket.py')\n"
    'expanded = importlib.util.module_from_spec(spec)\n'
    'spec.loader.exec_module(expanded)\n'
    '\n'
    '\n'
    'def record_key(row):\n'
    "    return '/'.join(str(int(row[k])) for k in KEYS)\n"
    '\n'
    '\n'
    'def sha256(path):\n'
    '    digest = hashlib.sha256()\n'
    "    with path.open('rb') as handle:\n"
    "        for block in iter(lambda: handle.read(1024 * 1024), b''):\n"
    '            digest.update(block)\n'
    '    return digest.hexdigest()\n'
    '\n'
    '\n'
    'def review_sample(reps, count=80):\n'
    '    """Simple random sample of pairs, independent of separation and flags."""\n'
    '    ordered = reps.sort_values(KEYS).reset_index(drop=True)\n'
    '    if ordered.duplicated(KEYS).any():\n'
    "        raise ValueError('Duplicate coverage pairs')\n"
    '    indices = np.random.default_rng(SEED).choice(len(ordered), min(count, len(ordered)), replace=False)\n'
    '    return ordered.iloc[indices].copy()\n'
    '\n'
    '\n'
    'def assess_labels(sample, labels):\n'
    '    """An independent source and reviewer are required even for unresolved labels."""\n'
    '    allowed = {record_key(row) for _, row in sample.iterrows()}\n'
    '    if set(labels) - allowed:\n'
    "        raise ValueError('Assignment labels contain keys outside the frozen random sample')\n"
    '    counts = Counter()\n'
    '    for key, label in labels.items():\n'
    "        if label.get('verdict') not in {'confirmed', 'incorrect', 'unresolved'}:\n"
    "            raise ValueError('Invalid assignment verdict: ' + key)\n"
    "        if label.get('evidence_type') not in {'film', 'independent charting'}:\n"
    "            raise ValueError('Tracking-only annotations cannot verify assignments: ' + key)\n"
    "        if not all(str(label.get(field, '')).strip() for field in ['reviewer', 'source', 'note']):\n"
    "            raise ValueError('Review requires reviewer, source locator, and note: ' + key)\n"
    "        counts[label['verdict']] += 1\n"
    '    total = len(sample)\n'
    "    return {'sampled': total, 'reviewed': len(labels), 'pending': total - len(labels),\n"
    "            'confirmed': counts['confirmed'], 'incorrect': counts['incorrect'],\n"
    "            'unresolved': counts['unresolved'],\n"
    "            'confirmed_share_of_sample': counts['confirmed'] / total if total and len(labels) == total else None,\n"
    "            'status': 'Independent labels pending' if not labels else 'Independent source annotations supplied; not independently audited',\n"
    "            'sampling': 'Simple random sample of accepted pairs, seed 20261022; no outcome-based selection',\n"
    "            'scope': 'Accepted pairs only; does not measure missed assignments or excluded plays. Shared players and plays limit independence.'}\n"
    '\n'
    '\n'
    'def phase_variant(frame, smoothing=3, heading_window=.4, flank=.6):\n'
    '    """Sensitivity detector using elapsed-time heading interpolation, not row offsets.\n'
    '\n'
    '    No coverage outcomes are used. This is a geometric turn, not a reaction-time\n'
    '    or biomechanical ground-truth annotation.\n'
    '    """\n'
    "    f = frame.sort_values('time')\n"
    '    if len(f) < 15 or f.time.duplicated().any():\n'
    '        return None\n'
    '    t = (pd.to_datetime(f.time) - pd.Timestamp(f.time.iloc[0])).dt.total_seconds().to_numpy()\n'
    "    raw = f[['s', 'dir', 'x', 'y']].to_numpy(dtype=float)\n"
    '    if not np.isfinite(raw).all() or not np.isfinite(t).all() or np.any(np.diff(t) <= 0) or np.diff(t).max() > .201:\n'
    '        return None\n'
    '    speed = f.s.rolling(smoothing, center=True, min_periods=1).median().to_numpy()\n'
    '    angle = np.deg2rad(f.dir.to_numpy())\n'
    '    sin = pd.Series(np.sin(angle)).rolling(smoothing, center=True, min_periods=1).mean()\n'
    '    cos = pd.Series(np.cos(angle)).rolling(smoothing, center=True, min_periods=1).mean()\n'
    '    heading = np.unwrap(np.arctan2(sin, cos).to_numpy())\n'
    '    half = heading_window / 2\n'
    '    delta = np.interp(t + half, t, heading) - np.interp(t - half, t, heading)\n'
    '    swing = np.abs(np.arctan2(np.sin(delta), np.cos(delta)))\n'
    '    eligible = (t >= max(flank, half)) & (t <= t[-1] - max(flank, half) + 1e-8) & (speed >= 1)\n'
    '    if not eligible.any() or np.max(swing[eligible]) < 1e-6:\n'
    '        return None\n'
    '    # Break numerical ties by earliest time; a coordinate rotation must not move\n'
    '    # an otherwise identical flat maximum because of floating-point roundoff.\n'
    '    maximum = np.max(swing[eligible])\n'
    '    center = t[np.flatnonzero(eligible & (swing >= maximum - 1e-10))[0]]\n'
    '    turn = np.flatnonzero(np.abs(t - center) <= .2 + 1e-8)\n'
    '    low = turn[np.argmin(speed[turn])]\n'
    '    before = np.flatnonzero((t >= center - flank - 1e-8) & (t <= t[low] - .1 + 1e-8))\n'
    '    after = np.flatnonzero((t >= t[low] + .1 - 1e-8) & (t <= center + flank + 1e-8))\n'
    '    if not len(before) or not len(after):\n'
    '        return None\n'
    '    pre = before[np.argmax(speed[before])]\n'
    '    post = after[np.argmax(speed[after])]\n'
    "    return {'center_seconds': float(center), 'turn_speed': float(speed[low]),\n"
    "            'heading_change_degrees': float(np.rad2deg(swing[np.argmin(abs(t - center))])),\n"
    "            'braking_loss': float((speed[pre] - speed[low]) / (t[low] - t[pre])),\n"
    "            'reacceleration': float((speed[post] - speed[low]) / (t[post] - t[low]))}\n"
    '\n'
    '\n'
    'def safe_rank(a, b):\n'
    '    if len(a) < 3 or pd.Series(a).nunique() < 2 or pd.Series(b).nunique() < 2:\n'
    '        return None\n'
    '    return float(pd.Series(a).rank().corr(pd.Series(b).rank()))\n'
    '\n'
    '\n'
    'def baseline_benchmarks(players):\n'
    '    """Fixed simple comparators, not an expanded sensor-feature model search."""\n'
    "    p = players[players.reps >= 10].sort_values('nfl_id').copy()\n"
    '    if len(p) < 12:\n'
    "        return {'players': len(p), 'status': 'Insufficient players', 'comparisons': []}\n"
    '    target = p.mean_separation.to_numpy()\n'
    '    configurations = [\n'
    "        ('Training-player median', [], 'No held-out-player measurements'),\n"
    "        ('Conventional combine only', ['forty', 'combine_height', 'combine_weight'], 'Pre-draft inputs; retrospective evaluation'),\n"
    "        ('Starting cushion only', ['snap_cushion'], 'Observed NFL context; not available pre-draft'),\n"
    "        ('Existing context baseline', ['snap_cushion', 'forty', 'combine_height', 'combine_weight'], 'Observed NFL context plus conventional combine')]\n"
    '    results = []\n'
    '    for label, columns, scope in configurations:\n'
    '        errors = []\n'
    '        for i in range(len(p)):\n'
    '            keep = np.arange(len(p)) != i\n'
    '            prediction = (expanded.ridge_predict(p[columns].to_numpy()[keep], target[keep], p[columns].to_numpy()[[i]])[0]\n'
    '                          if columns else np.median(target[keep]))\n'
    '            errors.append(abs(target[i] - prediction))\n'
    "        results.append({'name': label, 'inputs': columns, 'mae': float(np.mean(errors)), 'scope': scope})\n"
    "    return {'players': len(p), 'comparisons': results,\n"
    "            'status': 'Exploratory leave-one-player-out comparators on the same workload-eligible cohort',\n"
    "            'method': 'Player-average separation; equal held-out-player weight; fold-local imputation/scaling; fixed ridge penalty 1 where applicable.',\n"
    "            'limits': 'No untouched cohort, independent assignment truth, or full uncertainty interval. Comparing these baselines does not validate pre-draft prediction.'}\n"
    '\n'
    '\n'
    'def assess_phase_labels(review, labels):\n'
    "    prepared = {'/'.join(str(r[k]) for k in ['nfl_id', 'drill', 'event_id', 'attempt']): r for r in review}\n"
    '    if set(labels) - set(prepared):\n'
    "        raise ValueError('Phase annotation outside the random review sample')\n"
    '    confirmed, unresolved, inside, distances = 0, 0, 0, []\n'
    '    for key, label in labels.items():\n'
    "        if label.get('verdict') not in {'annotated', 'unresolved'}:\n"
    "            raise ValueError('Invalid phase annotation verdict: ' + key)\n"
    "        if label.get('evidence_type') not in {'film', 'independent charting'} or not all(\n"
    "                str(label.get(field, '')).strip() for field in ['reviewer', 'source', 'note']):\n"
    "            raise ValueError('Phase annotations require independent source, reviewer, and note')\n"
    "        if label['verdict'] == 'unresolved':\n"
    '            unresolved += 1\n'
    '            continue\n'
    "        start, end = float(label['phase_start_seconds']), float(label['phase_end_seconds'])\n"
    '        r = prepared[key]\n'
    "        if not np.isfinite([start, end]).all() or not 0 <= start <= end <= max(r['seconds']):\n"
    "            raise ValueError('Phase boundaries must be finite and within the observed attempt')\n"
    '        confirmed += 1\n'
    "        center = r['detected_center']\n"
    '        if center is not None:\n'
    '            distance = max(start - center, center - end, 0)\n'
    '            distances.append(distance)\n'
    '            inside += int(distance == 0)\n'
    "    return {'independently_annotated_phases': confirmed, 'unresolved': unresolved,\n"
    "            'pending': len(review) - len(labels), 'detected_annotated_phases': len(distances),\n"
    "            'detector_centers_inside_annotated_phase': inside,\n"
    "            'median_distance_to_phase_seconds': float(np.median(distances)) if distances else None,\n"
    "            'status': 'Independent annotations pending' if not labels else 'Source annotations supplied; source evidence not independently audited'}\n"
    '\n'
    '\n'
    'def phase_audit(drill, cb_ids):\n'
    "    selected = drill[drill.nfl_id.isin(cb_ids) & (drill.entity_type == 'PLAYER') & drill.drill_name.isin(expanded.DRILLS)]\n"
    "    settings = [('Elapsed-time reference', 3, .4, .6), ('No smoothing', 1, .4, .6),\n"
    "                ('Five-frame smoothing', 5, .4, .6), ('0.2-second heading window', 3, .2, .6),\n"
    "                ('0.6-second heading window', 3, .6, .6), ('0.4-second flanks', 3, .4, .4),\n"
    "                ('0.8-second flanks', 3, .4, .8), ('Coordinate-derived heading', 3, .4, .6)]\n"
    '    rows, packet = [], []\n'
    "    for key, frame in selected.groupby(['nfl_id', 'drill_name', 'event_id', 'attempt'], sort=True):\n"
    '        pid, name, event, attempt = key\n'
    '        base = expanded.phase_features(frame)\n'
    "        identity = {'nfl_id': int(pid), 'drill': name, 'event_id': str(event), 'attempt': int(attempt)}\n"
    '        if base is not None:\n'
    "            rows.append({**identity, 'setting': 'Published detector', **base})\n"
    '        for label, smooth, window, flank in settings:\n'
    '            candidate = frame\n'
    "            if label == 'Coordinate-derived heading':\n"
    "                candidate = frame.sort_values('time').copy()\n"
    '                times = (pd.to_datetime(candidate.time) - pd.Timestamp(candidate.time.iloc[0])).dt.total_seconds().to_numpy()\n'
    '                if candidate.time.duplicated().any() or len(candidate) < 3:\n'
    '                    continue\n'
    '                vx = np.gradient(candidate.x.to_numpy(), times)\n'
    '                vy = np.gradient(candidate.y.to_numpy(), times)\n'
    "                candidate['dir'] = np.rad2deg(np.arctan2(vx, vy)) % 360\n"
    '            result = phase_variant(candidate, smooth, window, flank)\n'
    '            if result is not None:\n'
    "                rows.append({**identity, 'setting': label, **result})\n"
    '        # Outcome-independent random selection is performed below, not here.\n'
    '        packet.append((identity, frame, base))\n'
    '    values = pd.DataFrame(rows)\n'
    "    values.to_csv(OUT / 'phase-sensitivity.csv', index=False)\n"
    '    summary = []\n'
    "    ids = ['nfl_id', 'drill', 'event_id', 'attempt']\n"
    '    for name, label in expanded.DRILLS.items():\n'
    '        subset = values[values.drill == name]\n'
    "        base = subset[subset.setting == 'Published detector']\n"
    "        multi = base.groupby('nfl_id').size()\n"
    '        variants = []\n'
    '        for setting, _, _, _ in settings:\n'
    '            alt = subset[subset.setting == setting]\n'
    "            paired = base.merge(alt, on=ids, suffixes=('_base', '_alt'), validate='one_to_one')\n"
    '            ranks = {}\n'
    "            for feature in ['turn_speed', 'braking_loss', 'reacceleration']:\n"
    '                # Aggregate the identical intersection of attempts per player.\n'
    "                player = paired.groupby('nfl_id')[[feature + '_base', feature + '_alt']].median()\n"
    "                ranks[feature] = safe_rank(player[feature + '_base'].to_numpy(), player[feature + '_alt'].to_numpy())\n"
    '            shifts = abs(paired.center_seconds_base - paired.center_seconds_alt)\n'
    "            variants.append({'setting': setting, 'paired_attempts': len(paired), 'detected_attempts': len(alt),\n"
    "                             'changed_eligibility': len(base) + len(alt) - 2 * len(paired),\n"
    "                             'median_center_shift': float(shifts.median()) if len(shifts) else None,\n"
    "                             'centers_shifted_over_point_two': int((shifts > .200001).sum()),\n"
    "                             'player_rank_correlations': ranks})\n"
    "        summary.append({'drill': label, 'attempts': len(base), 'players': int(base.nfl_id.nunique()),\n"
    "                        'players_with_repeated_attempts': int((multi > 1).sum()), 'variants': variants})\n"
    '    # A small random attempt sample per drill supports independent phase annotation.\n'
    '    review = []\n'
    '    rng = np.random.default_rng(SEED)\n'
    '    for name in expanded.DRILLS:\n'
    "        candidates = [p for p in packet if p[0]['drill'] == name]\n"
    '        for index in rng.choice(len(candidates), min(6, len(candidates)), replace=False):\n'
    '            identity, frame, base = candidates[index]\n'
    "            f = frame.sort_values('time').copy()\n"
    '            seconds = (pd.to_datetime(f.time) - pd.Timestamp(f.time.iloc[0])).dt.total_seconds()\n'
    "            review.append({**identity, 'seconds': seconds.tolist(), 'speed': f.s.tolist(),\n"
    "                           'x': f.x.tolist(), 'y': f.y.tolist(), 'direction': f.dir.tolist(),\n"
    "                           'detected_center': base['center_seconds'] if base else None})\n"
    "    phase_manifest = {'input_sha256': sha256(ROOT / 'nfl-big-data-bowl-2027.zip'),\n"
    "                      'attempts': [{k: r[k] for k in ['nfl_id', 'drill', 'event_id', 'attempt']} for r in review]}\n"
    "    phase_manifest_path = OUT / 'phase-sample-manifest.json'\n"
    "    if phase_manifest_path.exists() and json.loads(phase_manifest_path.read_text(encoding='utf-8')) != phase_manifest:\n"
    "        raise ValueError('Frozen phase sample changed; archive prior validation outputs before a new study')\n"
    "    phase_manifest_path.write_text(json.dumps(phase_manifest, indent=2), encoding='utf-8')\n"
    "    (OUT / 'phase-review-panels.json').write_text(json.dumps(review, indent=2, allow_nan=False), encoding='utf-8')\n"
    "    pd.DataFrame([{k: r[k] for k in ['nfl_id', 'drill', 'event_id', 'attempt']} for r in review]).assign(\n"
    "        reviewer='', evidence_type='', source='', phase_start_seconds='', phase_end_seconds='', verdict='', note=''\n"
    "    ).to_csv(OUT / 'phase-review-template.csv', index=False)\n"
    "    label_path = OUT / 'phase-labels.json'\n"
    "    annotations = assess_phase_labels(review, json.loads(label_path.read_text(encoding='utf-8')) if label_path.exists() else {})\n"
    "    return {'status': 'Detector sensitivity measured; independent annotation status reported separately',\n"
    "            'annotation_review': annotations, 'prepared_review_attempts': len(review), 'drills': summary,\n"
    "            'scope': 'Rank correlation measures sensitivity, not accuracy or test-retest reliability. Entry speed and turn angle are not controlled.',\n"
    "            'settings': 'Published detector versus elapsed-time reference, six one-setting perturbations, and coordinate-derived heading; no outcome-based tuning.'}\n"
    '\n'
    '\n'
    'def application(reps):\n'
    '    """Prespecified descriptive film-review flags; no new success endpoint or model."""\n'
    '    pairs = reps.copy()\n'
    "    pairs['average_tight'] = pairs.mean_separation < 1\n"
    "    pairs['throw_tight'] = pairs.throw_separation < 1\n"
    "    pairs['review_reason'] = np.select(\n"
    '        [pairs.average_tight & ~pairs.throw_tight,\n'
    '         ~pairs.average_tight & pairs.throw_tight],\n'
    "        ['Average-tight but not throw-tight', 'Throw-tight despite average >=1 yard'], default='Ordinary context review')\n"
    '    cells = []\n'
    '    for mean in [True, False]:\n'
    '        for throw in [True, False]:\n'
    '            p = pairs[(pairs.average_tight == mean) & (pairs.throw_tight == throw)]\n'
    "            cells.append({'average': '<1 yard' if mean else '>=1 yard', 'throw': '<1 yard' if throw else '>=1 yard',\n"
    "                          'reps': len(p), 'players': int(p.nfl_id.nunique())})\n"
    '    strata = []\n'
    "    for label, mask in [('Under 2 yards', pairs.snap_cushion < 2),\n"
    "                        ('2 to under 3 yards', (pairs.snap_cushion >= 2) & (pairs.snap_cushion < 3)),\n"
    "                        ('3 to 5 yards', pairs.snap_cushion >= 3)]:\n"
    '        p = pairs[mask]\n'
    "        strata.append({'cushion': label, 'reps': len(p), 'average_tight': int(p.average_tight.sum()),\n"
    "                       'throw_tight': int(p.throw_tight.sum())})\n"
    "    queue = pairs[pairs.review_reason != 'Ordinary context review'].sort_values(KEYS)\n"
    "    queue.to_csv(OUT / 'football-review-queue.csv', index=False)\n"
    '    shortlist = pd.concat([p.sample(n=min(4, len(p)), random_state=SEED) for p in [\n'
    '        pairs[pairs.average_tight & ~pairs.throw_tight],\n'
    '        pairs[~pairs.average_tight & pairs.throw_tight & (pairs.snap_cushion >= 3)],\n'
    '        pairs[pairs.average_tight & pairs.throw_tight]]]).sort_values(KEYS)\n'
    "    shortlist.to_csv(OUT / 'football-meeting-shortlist.csv', index=False)\n"
    "    return {'use': 'Select contrasting clips for a coverage film meeting', 'cells': cells, 'cushion_strata': strata,\n"
    "            'late_gap_flags': int((pairs.average_tight & (pairs.throw_separation >= 2)).sum()),\n"
    "            'throw_tight_only_flags': int((~pairs.average_tight & pairs.throw_tight).sum()),\n"
    "            'average_tight_only_flags': int((pairs.average_tight & ~pairs.throw_tight).sum()),\n"
    "            'queued_pairs': len(queue), 'meeting_shortlist': len(shortlist),\n"
    "            'shortlist_policy': 'Up to four fixed-seed examples each: average-only tight; throw-only tight starting >=3 yards; both tight. Diagnostic examples, not an accuracy sample.',\n"
    "            'scope': 'Descriptive disagreement between measurements, not proven coverage wins, target suppression, causal value, or scouting grades.',\n"
    "            'threshold_policy': 'Keep Blanket <1 yard snap-to-throw; >=2 yards at throw is a separate review flag, not a replacement definition.'}\n"
    '\n'
    '\n'
    'def line_svg(x, y, caption, marker=None, equal_aspect=True):\n'
    '    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)\n'
    '    valid = np.isfinite(x) & np.isfinite(y)\n'
    '    if not valid.any():\n'
    "        return '<p>No finite coordinates</p>'\n"
    '    x, y = x[valid], y[valid]\n'
    '    sx, sy = 410 / max(float(np.ptp(x)), 1), 155 / max(float(np.ptp(y)), 1)\n'
    '    if equal_aspect:\n'
    '        sx = sy = min(sx, sy)\n'
    '    px, py = 60 + (x - x.min()) * sx, 180 - (y - y.min()) * sy\n'
    "    points = ' '.join(f'{a:.2f},{b:.2f}' for a, b in zip(px, py))\n"
    "    mark = ''\n"
    '    if marker is not None:\n'
    '        nearest = int(np.argmin(abs(x - marker)))\n'
    '        mark = f\'<circle cx="{px[nearest]:.2f}" cy="{py[nearest]:.2f}" r="6" fill="#ac4400"/>\'\n'
    "    ticks = ''\n"
    '    for value in np.linspace(x.min(), x.max(), 5):\n'
    '        position = 60 + (value - x.min()) * sx\n'
    '        ticks += f\'<text x="{position:.2f}" y="202" text-anchor="middle" font-size="11">{value:.1f}</text>\'\n'
    '    for value in np.linspace(y.min(), y.max(), 4):\n'
    '        position = 180 - (value - y.min()) * sy\n'
    '        ticks += f\'<text x="52" y="{position+4:.2f}" text-anchor="end" font-size="11">{value:.1f}</text>\'\n'
    '    return f\'<figure><svg viewBox="0 0 500 220" role="img" aria-label="{escape(caption, quote=True)}"><path d="M60 20V180H475" fill="none" stroke="#89988d"/>{ticks}<polyline points="{points}" fill="none" stroke="#176957" stroke-width="2"/>{mark}</svg><figcaption>{escape(caption)}</figcaption></figure>\'\n'
    '\n'
    '\n'
    'def assignment_panels(sample, tracking):\n'
    "    groups = {k: g for k, g in tracking.groupby(['game_id', 'play_id'], sort=False)}\n"
    '    panels = []\n'
    '    for index, (_, row) in enumerate(sample.iterrows(), 1):\n'
    '        t = groups[(row.game_id, row.play_id)]\n'
    "        d = t[t.nfl_id == row.nfl_id].sort_values('time')\n"
    "        snap = d[d.event == 'ball_snap'].iloc[0].time\n"
    "        end = d[d.event == 'pass_forward'].iloc[0].time\n"
    '        # Show every tracked player as context; no computed separation or outcome.\n'
    '        window = t[t.time.between(snap, end)].copy()\n'
    "        coords = window[['x', 'y']].to_numpy(dtype=float)\n"
    '        low = coords.min(axis=0)\n'
    '        span = np.maximum(coords.max(axis=0) - low, 1)\n'
    '        scale = min(680 / span[0], 300 / span[1])\n'
    '        paths = []\n'
    "        for pid, f in window.groupby('nfl_id'):\n"
    "            f = f.sort_values('time')\n"
    "            color = '#176957' if pid == row.nfl_id else '#ad6700' if pid == row.receiver_id else '#929999'\n"
    '            width = 3 if pid in [row.nfl_id, row.receiver_id] else 1\n'
    "            xy = (f[['x', 'y']].to_numpy() - low) * scale\n"
    "            points = ' '.join(f'{x+20:.2f},{330-y:.2f}' for x, y in xy)\n"
    '            x, y = xy[0]\n'
    '            paths.append(f\'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="{width}"/><circle cx="{x+20:.2f}" cy="{330-y:.2f}" r="4" fill="{color}"/><text x="{x+25:.2f}" y="{325-y:.2f}" font-size="11">{int(pid)}</text>\')\n'
    '        panels.append(f\'<article><h2>A{index:03}: {escape(record_key(row))}</h2><p>Green: CB; amber: proposed receiver; gray: other tracked players. Dots: snap. Equal coordinate scales.</p><svg viewBox="0 0 720 350" role="img" aria-label="All available tracked paths for assignment review">{"".join(paths)}</svg></article>\')\n'
    '    return panels\n'
    '\n'
    '\n'
    'def write_html(name, title, intro, articles):\n'
    '    html = \'<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\'\n'
    "    html += f'<title>{escape(title)}</title><style>body{{font:16px/1.6 system-ui;max-width:1000px;margin:30px auto;padding:20px;background:#f4f5ee;color:#172d27}}article{{border-top:1px solid #bac9bf;padding:25px 0}}svg{{width:100%;max-height:400px}}figure{{margin:15px 0}}figcaption{{font-size:14px}}code{{overflow-wrap:anywhere}}</style>'\n"
    "    html += f'<h1>{escape(title)}</h1><p>{escape(intro)}</p>' + ''.join(articles) + '</html>'\n"
    "    (OUT / name).write_text(html, encoding='utf-8')\n"
    '\n'
    '\n'
    'def main():\n'
    '    parser = argparse.ArgumentParser(description=__doc__)\n'
    "    parser.add_argument('--publish', action='store_true')\n"
    '    args = parser.parse_args()\n'
    '    OUT.mkdir(parents=True, exist_ok=True)\n'
    "    source = json.loads((INPUT / 'summary.json').read_text(encoding='utf-8'))\n"
    "    archive_path = ROOT / 'nfl-big-data-bowl-2027.zip'\n"
    "    if sha256(archive_path) != source['input_sha256']:\n"
    "        raise ValueError('Archive changed; rerun expand-blanket.py first')\n"
    "    cache = json.loads((INPUT / 'tracking-cache.json').read_text(encoding='utf-8'))\n"
    "    if cache['input_sha256'] != source['input_sha256']:\n"
    "        raise ValueError('Tracking cache does not match source archive')\n"
    "    reps = pd.read_csv(INPUT / 'coverage-reps.csv')\n"
    "    if (len(reps) != source['screened_reps'] or int(reps.blanket.sum()) != source['provisional_blankets']\n"
    "            or reps.nfl_id.nunique() != source['screened_players']\n"
    "            or not np.isclose(reps.mean_separation.median(), source['median_separation'])):\n"
    "        raise ValueError('Coverage source differs from expanded summary')\n"
    "    if not np.isfinite(reps[['mean_separation', 'throw_separation', 'snap_cushion']].to_numpy()).all():\n"
    "        raise ValueError('Nonfinite coverage measurements')\n"
    '    if not (reps.blanket == (reps.mean_separation < 1)).all():\n'
    "        raise ValueError('Coverage flags differ from the strict Blanket definition')\n"
    '    sample = review_sample(reps)\n'
    "    signature = {'source_sha256': sha256(INPUT / 'coverage-reps.csv'), 'seed': SEED,\n"
    "                 'sample_keys': [record_key(row) for _, row in sample.iterrows()]}\n"
    "    manifest_path = OUT / 'assignment-sample-manifest.json'\n"
    "    if manifest_path.exists() and json.loads(manifest_path.read_text(encoding='utf-8')) != signature:\n"
    "        raise ValueError('Frozen assignment sample changed; archive the old validation directory before a new study')\n"
    "    manifest_path.write_text(json.dumps(signature, indent=2), encoding='utf-8')\n"
    '    worksheet = sample[KEYS].copy()\n'
    "    worksheet['review_id'] = [f'A{i+1:03}' for i in range(len(worksheet))]\n"
    "    for field in ['verdict', 'reviewer', 'evidence_type', 'source', 'note']:\n"
    "        worksheet[field] = ''\n"
    "    worksheet.to_csv(OUT / 'assignment-review-template.csv', index=False)\n"
    "    labels_path = OUT / 'assignment-labels.json'\n"
    "    labels = json.loads(labels_path.read_text(encoding='utf-8')) if labels_path.exists() else {}\n"
    '    assignment = assess_labels(sample, labels)\n'
    '    positives = reps[reps.blanket].sort_values(KEYS).copy()\n'
    '    positive_template = positives[KEYS].copy()\n'
    "    for field in ['verdict', 'reviewer', 'evidence_type', 'source', 'note']:\n"
    "        positive_template[field] = ''\n"
    "    positive_template.to_csv(OUT / 'positive-review-template.csv', index=False)\n"
    "    positive_path = OUT / 'positive-assignment-labels.json'\n"
    "    positive_review = assess_labels(positives, json.loads(positive_path.read_text(encoding='utf-8')) if positive_path.exists() else {})\n"
    "    positive_review['sampling'] = 'Every provisional Blanket, reported separately from the random accuracy sample; samples may overlap'\n"
    "    assignment['positive_review'] = positive_review\n"
    "    tracking = pd.read_pickle(INPUT / 'tracking.pkl')\n"
    "    write_html('assignment-review.html', 'Blanket: random assignment review',\n"
    "               'Private, outcome-masked random sample. Use independent film or charting before labeling. Only selected players are tracked; unseen players may hold the true assignment. Review against source video, not trajectory resemblance.',\n"
    '               assignment_panels(sample, tracking))\n'
    "    write_html('positive-assignment-review.html', 'Blanket: all provisional positives',\n"
    "               'Private outcome-enriched review, separate from the random sample. Review each proposed assignment with independent evidence. These cases cannot estimate overall pairing accuracy.',\n"
    '               assignment_panels(positives, tracking))\n'
    '    with ZipFile(archive_path) as archive:\n'
    "        prefix = 'nfl-big-data-bowl-2027/'\n"
    "        cb = pd.read_csv(archive.open(prefix + 'players.csv'))\n"
    "        drill = pd.read_csv(archive.open(prefix + 'combine_tracking.csv'))\n"
    "        members = [n for n in archive.namelist() if not n.endswith('/')]\n"
    "        metadata_columns = pd.read_csv(archive.open(prefix + 'player_play.csv'), nrows=0).columns.tolist()\n"
    "        inventory = {'csv_files': sum(n.lower().endswith('.csv') for n in members),\n"
    "                     'video_files': sum(Path(n).suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv', '.webm'} for n in members),\n"
    "                     'coverage_label_fields_present': all(n in metadata_columns for n in ['coverage_assignment', 'coverage_assignment_at_snap']),\n"
    "                     'explicit_receiver_id_field_present': any(n in metadata_columns for n in ['receiver_id', 'assigned_receiver_id', 'coverage_receiver_id', 'matchup_nfl_id']),\n"
    "                     'interpretation': 'The supplied archive contains CSV tracking and metadata, not film. MAN labels do not name the assigned receiver.'}\n"
    "    phases = phase_audit(drill, cb.loc[cb.nfl_position == 'CB', 'nfl_id'])\n"
    "    phase_panels = json.loads((OUT / 'phase-review-panels.json').read_text(encoding='utf-8'))\n"
    '    articles = []\n'
    '    for r in phase_panels:\n'
    '        title = f"{r[\'drill\']} / {r[\'nfl_id\']} / {r[\'event_id\']} / attempt {r[\'attempt\']}"\n'
    "        articles.append('<article><h2>' + escape(title) + '</h2>' + line_svg(r['x'], r['y'], 'Trajectory: equal source-coordinate scales') +\n"
    "                        line_svg(r['seconds'], r['speed'], 'Horizontal: time (seconds); vertical: speed (source units); orange dot is detector center', r['detected_center'], equal_aspect=False) + '</article>')\n"
    "    write_html('phase-review.html', 'Blanket: drill phase review',\n"
    "               'Private random sample, six attempts per drill. Compare the detector with independent drill video or annotated phase boundaries. A plausible trajectory is not independent phase verification.', articles)\n"
    '    football = application(reps)\n'
    "    benchmarks = baseline_benchmarks(pd.read_csv(INPUT / 'players.csv'))\n"
    "    queue = pd.read_csv(OUT / 'football-meeting-shortlist.csv')\n"
    '    # Public flags are aggregate only. The local casebook contains the review identifiers.\n'
    "    write_html('football-review.html', 'Blanket: coverage film meeting queue',\n"
    "               'Start by confirming the assignment from independent film. Then compare starting cushion, average separation, and throw separation. These flags prioritize questions; they do not grade defenders.',\n"
    "               [f'<article><h2>{escape(record_key(row))}</h2><p>{escape(row.review_reason)}</p><p>Cushion {row.snap_cushion:.2f}; average {row.mean_separation:.2f}; throw {row.throw_separation:.2f} assumed yards. Route: {escape(str(row.route))}.</p><ol><li>Is this the assigned receiver?</li><li>Does the starting cushion explain the disagreement?</li><li>Did the receiver create separation late, or did the CB close it?</li></ol></article>' for _, row in queue.iterrows()] + assignment_panels(queue, tracking))\n"
    "    report = {'validation_version': 1, 'updated': 'October 22, 2026', 'input_sha256': source['input_sha256'],\n"
    "              'coverage_source_sha256': signature['source_sha256'], 'source_analysis_version': source['analysis_version'],\n"
    "              'audit_code_sha256': sha256(Path(__file__)),\n"
    "              'detector_code_sha256': sha256(ROOT / 'tools/expand-blanket.py'),\n"
    "              'screened_reps': len(reps), 'assignment': assignment, 'phases': phases, 'application': football,\n"
    "              'archive_inventory': inventory,\n"
    "              'baselines': benchmarks, 'player_source_sha256': sha256(INPUT / 'players.csv'),\n"
    "              'units': 'Game distances assume yards; combine speed retains source units. Current-year field dictionary confirmation remains pending.',\n"
    "              'model_policy': 'Version 4 sensor comparisons remain unchanged. Four fixed simple baselines were added; no sensor-feature model was tuned or selected.'}\n"
    "    (OUT / 'summary.json').write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')\n"
    '    if args.publish:\n'
    "        published = json.loads((ROOT / '_data/blanket_expanded.json').read_text(encoding='utf-8'))\n"
    "        if published['input_sha256'] != source['input_sha256'] or published['screened_reps'] != len(reps):\n"
    "            raise ValueError('Public expanded snapshot differs from validation source')\n"
    "        (ROOT / '_data/blanket_validation.json').write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')\n"
    "    print(json.dumps({'assignment': assignment, 'phase_review_attempts': len(phase_panels), 'application': football}, indent=2))\n"
    '\n'
    '\n'
    "if __name__ == '__main__':\n"
    '    main()\n'
)
assert hashlib.sha256(source.encode('utf-8')).hexdigest() == '5004a99cede73dd16d147825a32cfbb185b31d4daf89748eae06ab087d97bf77'
(WORK / 'tools' / 'validate-blanket.py').write_bytes(source.encode('utf-8'))


In [ ]:
# plot-blanket-models.py; SHA-256 of UTF-8 source: 97899fffa2efe4ec7ce27a670abcdb45fb78d9b7681252074eb865333dc4d31a
source = (
    '"""Plot public aggregate held-out-player model errors, never player trajectories."""\n'
    'from pathlib import Path\n'
    'import os\n'
    'import sys\n'
    'import json\n'
    '\n'
    'ROOT=Path(__file__).resolve().parents[1]\n'
    "os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'local-data/big-data-bowl/expanded/matplotlib-cache'))\n"
    'try:\n'
    '    import matplotlib\n'
    'except ImportError:\n'
    "    sys.path.insert(0,str(ROOT/'local-data/big-data-bowl/plot-deps'))\n"
    '    import matplotlib\n'
    "matplotlib.use('Agg')\n"
    'import matplotlib.pyplot as plt\n'
    'import numpy as np\n'
    '\n'
    "data=json.loads((ROOT/'_data/blanket_expanded.json').read_text(encoding='utf-8'))\n"
    "models=data['models']\n"
    "labels=[m['drill'].replace(' / ','/')+' · '+m['feature'].replace('_',' ') for m in models]\n"
    "estimate=np.array([m['mae_improvement'] for m in models])\n"
    "lower=np.array([m['improvement_95'][0] for m in models])\n"
    "upper=np.array([m['improvement_95'][1] for m in models])\n"
    "plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,'axes.spines.right':False,'svg.hashsalt':'blanket-v4'})\n"
    "fig,ax=plt.subplots(figsize=(10,6),layout='constrained')\n"
    "fig.patch.set_facecolor('#f4f5ee');ax.set_facecolor('#f4f5ee')\n"
    'y=np.arange(len(models))\n'
    "ax.errorbar(estimate,y,xerr=[estimate-lower,upper-estimate],fmt='o',color='#245b49',capsize=4,lw=1.6,markersize=5)\n"
    "ax.axvline(0,color='#6a7169',ls='--',lw=1)\n"
    "ax.set_yticks(y,labels);ax.invert_yaxis();ax.grid(axis='x',alpha=.2)\n"
    "ax.set_xlabel('Reduction in held-out-player MAE (yards): positive favors the added feature',labelpad=12)\n"
    "ax.set_title('Do drill features improve the context baseline?',loc='left',fontweight='bold',pad=37,fontsize=15)\n"
    "ax.text(0,1.04,'26 CBs · 9 exploratory comparisons · bars resample held-out error differences',transform=ax.transAxes,fontsize=10,color='#52665e')\n"
    "for split in [2.5,5.5]:ax.axhline(split,color='#d9dfd1',lw=1)\n"
    "destination=ROOT/'assets/images/blanket'\n"
    'destination.mkdir(parents=True,exist_ok=True)\n'
    "for ext in ['png','svg']:\n"
    "    target=destination/f'held-out-models.{ext}'\n"
    "    fig.savefig(target,dpi=160,metadata={'Date':None} if ext=='svg' else {})\n"
    "    if ext=='svg':\n"
    "        target.write_text('\\n'.join(line.rstrip() for line in target.read_text(encoding='utf-8').splitlines())+'\\n',encoding='utf-8')\n"
    'plt.close(fig)\n'
    "print(destination/'held-out-models.png')\n"
)
assert hashlib.sha256(source.encode('utf-8')).hexdigest() == '97899fffa2efe4ec7ce27a670abcdb45fb78d9b7681252074eb865333dc4d31a'
(WORK / 'tools' / 'plot-blanket-models.py').write_bytes(source.encode('utf-8'))


In [ ]:
# plot-blanket-validation.py; SHA-256 of UTF-8 source: 803bb71d5ae773df2da9d59cc08ba193cf1afb87e567ae51adf982cc649ae939
source = (
    '"""Render public aggregate measurement audits; never export individual tracking."""\n'
    'import json\n'
    'import os\n'
    'import sys\n'
    'from pathlib import Path\n'
    '\n'
    'ROOT = Path(__file__).resolve().parents[1]\n'
    "os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'local-data/big-data-bowl/expanded/matplotlib-cache'))\n"
    'try:\n'
    '    import matplotlib\n'
    'except ImportError:\n'
    "    sys.path.insert(0, str(ROOT / 'local-data/big-data-bowl/plot-deps'))\n"
    '    import matplotlib\n'
    "matplotlib.use('Agg')\n"
    'import matplotlib.pyplot as plt\n'
    'import numpy as np\n'
    '\n'
    "data = json.loads((ROOT / '_data/blanket_validation.json').read_text(encoding='utf-8'))\n"
    "plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,\n"
    "                     'axes.spines.top': False, 'axes.spines.right': False, 'svg.hashsalt': 'blanket-validation-v1'})\n"
    "destination = ROOT / 'assets/images/blanket'\n"
    'destination.mkdir(parents=True, exist_ok=True)\n'
    '\n'
    '\n'
    'def save(fig, name):\n'
    "    for ext in ['png', 'svg']:\n"
    "        target = destination / f'{name}.{ext}'\n"
    "        fig.savefig(target, dpi=160, metadata={'Date': None} if ext == 'svg' else {})\n"
    "        if ext == 'svg':\n"
    "            target.write_text('\\n'.join(line.rstrip() for line in target.read_text(encoding='utf-8').splitlines()) + '\\n', encoding='utf-8')\n"
    '    plt.close(fig)\n'
    '\n'
    '\n'
    "fig, ax = plt.subplots(figsize=(10, 6.5), layout='constrained')\n"
    "fig.patch.set_facecolor('#f4f5ee')\n"
    'rows, labels = [], []\n'
    "for drill in data['phases']['drills']:\n"
    "    variants = [v for v in drill['variants'] if v['setting'] != 'Elapsed-time reference']\n"
    "    for feature in ['turn_speed', 'braking_loss', 'reacceleration']:\n"
    "        rows.append([v['player_rank_correlations'][feature] for v in variants])\n"
    "        labels.append(drill['drill'].replace(' / ', '/') + ' | ' + feature.replace('_', ' '))\n"
    'matrix = np.asarray([[np.nan if v is None else v for v in row] for row in rows])\n'
    "im = ax.imshow(matrix, vmin=-1, vmax=1, cmap='BrBG', aspect='auto')\n"
    'ax.set_yticks(range(len(labels)), labels)\n'
    "ax.set_xticks(range(7), ['No\\nsmoothing', '5-frame\\nsmoothing', '0.2 s\\nheading', '0.6 s\\nheading', '0.4 s\\nflanks', '0.8 s\\nflanks', 'x/y-derived\\nheading'], fontsize=9)\n"
    'for i, row in enumerate(matrix):\n'
    '    for j, v in enumerate(row):\n'
    "        ax.text(j, i, 'N/A' if np.isnan(v) else f'{v:.2f}', ha='center', va='center',\n"
    "                color='white' if abs(v) > .65 else '#172d27', fontsize=10)\n"
    'for split in [2.5, 5.5]:\n'
    "    ax.axhline(split, color='white', lw=2)\n"
    "ax.set_title('How much do movement-feature rankings depend on the detector?', loc='left', fontweight='bold', pad=30, fontsize=13)\n"
    "fig.suptitle('Spearman correlation with published player-median features; 60 CBs per drill\\nSensitivity and geometric agreement are not independent phase validation.', fontsize=10, color='#52665e')\n"
    "fig.colorbar(im, ax=ax, shrink=.7, label='Player-rank correlation')\n"
    "save(fig, 'phase-sensitivity')\n"
    '\n'
    "fig, axes = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', gridspec_kw={'width_ratios': [1, 1.3]})\n"
    "fig.patch.set_facecolor('#f4f5ee')\n"
    'ax = axes[0]\n'
    "matrix = np.array([c['reps'] for c in data['application']['cells']]).reshape(2, 2)\n"
    "ax.imshow(matrix, cmap='Greens', vmin=0, vmax=max(matrix.ravel()))\n"
    "ax.set_xticks([0, 1], ['<1 yard', '>=1 yard'])\n"
    "ax.set_yticks([0, 1], ['<1 yard', '>=1 yard'])\n"
    "ax.set_xlabel('Separation at throw')\n"
    "ax.set_ylabel('Average snap-to-throw separation')\n"
    "ax.set_title('Two measurements, different questions', fontsize=11, pad=15)\n"
    'for i in range(2):\n'
    '    for j in range(2):\n'
    "        ax.text(j, i, str(matrix[i, j]), ha='center', va='center', fontsize=22, fontweight='bold',\n"
    "                color='white' if matrix[i, j] > 300 else '#172d27')\n"
    'ax = axes[1]\n'
    "strata = data['application']['cushion_strata']\n"
    'x = np.arange(3)\n'
    "for offset, field, label, color in [(-.18, 'average_tight', 'Average <1 yard', '#245b49'), (.18, 'throw_tight', 'Throw <1 yard', '#a66a16')]:\n"
    "    rates = [100 * s[field] / s['reps'] for s in strata]\n"
    '    bars = ax.bar(x + offset, rates, .34, label=label, color=color)\n'
    '    ax.bar_label(bars, labels=[f"{s[field]}/{s[\'reps\']}" for s in strata], fontsize=8, padding=3)\n'
    "ax.set_xticks(x, ['<2 yd', '2 to <3 yd', '3 to 5 yd'])\n"
    "ax.set_xlabel('Starting cushion')\n"
    "ax.set_ylabel('Share of measured pairs (%)')\n"
    'ax.set_ylim(0, 55)\n'
    "ax.set_title('Starting alignment changes interpretation', fontsize=11, pad=15)\n"
    "ax.legend(loc='upper right', fontsize=8)\n"
    'fig.suptitle(f"{data[\'screened_reps\']} inferred pairs | Descriptive only; assignments remain unverified", fontsize=11)\n'
    "save(fig, 'coverage-context')\n"
    "print(destination / 'phase-sensitivity.png')\n"
    "print(destination / 'coverage-context.png')\n"
)
assert hashlib.sha256(source.encode('utf-8')).hexdigest() == '803bb71d5ae773df2da9d59cc08ba193cf1afb87e567ae51adf982cc649ae939'
(WORK / 'tools' / 'plot-blanket-validation.py').write_bytes(source.encode('utf-8'))


## Prepare licensed input and reproduce

An existing ZIP is linked where possible, otherwise copied. Extracted CSVs
are streamed into a canonical local ZIP without loading them all into memory.
For repeated **local** runs only, `BLANKET_USE_VERIFIED_CACHE=1` can reuse the
repository's tracking cache. The analysis checks its archive hash and filter
version before accepting it. This is a cached rerun, not a cold-data audit.

Expect several minutes and substantial disk use on a fresh run. Source
evidence labels are absent by default, so independent reviews correctly
appear pending. No package installations or web requests are made.


In [ ]:
required = ['players.csv', 'combine_results.csv', 'combine_tracking.csv', 'player_play.csv',
            'games.csv', 'player_career_successes.csv',
            'game_tracking_2023.csv', 'game_tracking_2024.csv', 'game_tracking_2025.csv']
archive = WORK / 'nfl-big-data-bowl-2027.zip'
if DATA_PATH.is_dir():
    missing = [name for name in required if not (DATA_PATH / name).is_file()]
    if missing:
        raise FileNotFoundError('Missing official CSV files: ' + ', '.join(missing))
    # Rebuild from the supplied folder; do not silently reuse a stale ZIP.
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as handle:
        for name in required:
            handle.write(DATA_PATH / name, 'nfl-big-data-bowl-2027/' + name)
elif DATA_PATH.is_file():
    if archive.exists() and not os.path.samefile(DATA_PATH, archive):
        raise RuntimeError('Working ZIP already exists and is not this input. Choose a fresh WORK directory.')
    if not archive.exists():
        try:
            os.link(DATA_PATH, archive)
        except OSError:
            shutil.copy2(DATA_PATH, archive)
else:
    raise FileNotFoundError(DATA_PATH)
use_cache = os.environ.get('BLANKET_USE_VERIFIED_CACHE') == '1'
if use_cache and repository:
    cache = WORK / 'local-data/big-data-bowl/expanded'
    cache.mkdir(parents=True, exist_ok=True)
    for filename in ['tracking.pkl', 'tracking-cache.json']:
        original = repository / 'local-data/big-data-bowl/expanded' / filename
        target = cache / filename
        if original.exists() and not target.exists():
            shutil.copy2(original, target)
for filename in ['expand-blanket.py', 'validate-blanket.py', 'plot-blanket-models.py', 'plot-blanket-validation.py']:
    arguments = [sys.executable, str(WORK / 'tools' / filename)]
    if filename in ['expand-blanket.py', 'validate-blanket.py']:
        arguments.append('--publish')
    run = subprocess.run(arguments, cwd=WORK, text=True, capture_output=True)
    (WORK / (filename + '.log')).write_text(run.stdout + run.stderr, encoding='utf-8')
    if run.returncode:
        raise RuntimeError(filename + ' failed; inspect its private log. ' + run.stderr[-1500:])
    print('Completed:', filename)


## Aggregate results

Blanket stays strictly below one yard averaged from snap to throw. Window
comparisons are alternatives, not revised definitions. The baseline table
uses the same workload-eligible held-out players. NFL cushion is contextual
information, not an available pre-draft input. Rank sensitivity is not phase
truth. Error resampling holds predictions fixed and omits full training and
assignment uncertainty. No untouched test cohort exists.


In [ ]:
import pandas as pd
expanded = json.loads((WORK / '_data/blanket_expanded.json').read_text(encoding='utf-8'))
audit = json.loads((WORK / '_data/blanket_validation.json').read_text(encoding='utf-8'))
assert expanded['screened_reps'] == sum(c['reps'] for c in audit['application']['cells'])
assert expanded['candidate_reps'] == expanded['screened_reps'] + sum(expanded['exclusions'].values())
print('Measured pairs:', expanded['screened_reps'], '| CBs:', expanded['screened_players'],
      '| Provisional Blankets:', expanded['provisional_blankets'])
print('\nSimple comparators (MAE in yards):')
print(pd.DataFrame(audit['baselines']['comparisons'])[['name', 'mae', 'scope']].to_string(index=False))
print('\nAll nine historical sensor comparisons:')
print(pd.DataFrame(expanded['models'])[['drill', 'feature', 'players', 'baseline_mae',
                                       'augmented_mae', 'mae_improvement', 'improvement_95']].to_string(index=False))
print('\nIndependent assignment review:', audit['assignment']['status'])
print('Independent phase review:', audit['phases']['annotation_review']['status'])
print('Film-meeting shortlist:', audit['application']['meeting_shortlist'], 'private examples')


In [ ]:
figures = [WORK / 'assets/images/blanket' / name for name in
           ['coverage-context.png', 'phase-sensitivity.png', 'held-out-models.png']]
assert all(path.is_file() for path in figures)
try:
    from IPython.display import Image, display
except ImportError:
    print('Figures written:', *figures, sep='\n')
else:
    for path in figures:
        display(Image(filename=str(path)))


## Interpretation and submission status

This reproduces a retrospective exploratory analysis, not a validated
prospect forecast. The original sensor comparisons are preserved. Four
simple comparators clarify their interpretation; no sensor model or penalty
was tuned in this audit. Receiver assignments, physical phases, current-year
units, route/receiver adjustment, and genuinely new evaluation data remain
unresolved. Missing film labels are not evidence of incorrect assignments.

Raw tracking, player identities, caches, local review packets, and annotations
remain in the working directory; publish only intended aggregate findings.
The source notebook intentionally has no saved outputs. Execute it with the
official data to inspect the figures and tables. A GitHub notebook is not an
attached public Kaggle notebook: that separate competition step is pending.

Data attribution: NFL / Next Gen Stats, via
[NFL Big Data Bowl 2027](https://www.kaggle.com/competitions/nfl-big-data-bowl-2027).
